# 🏠 House Price Prediction — SkillCraft Task 1
**Goal:** Predict the sale price of houses using features like size, quality, location, etc.

This notebook walks through every step with explanations written for beginners.

---

## STEP 1 — Import Libraries

Before we do anything, we need to load the Python tools we'll use.
Think of these like apps we're opening before starting work.

In [ ]:
# pandas  → for loading and working with data (like Excel in Python)
import pandas as pd

# numpy   → for math operations on numbers
import numpy as np

# matplotlib & seaborn → for making charts and graphs
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn → the machine learning library
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

# This line makes graphs appear inside the notebook
%matplotlib inline

# Make graphs look nicer
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries loaded successfully!')

---
## STEP 2 — Load the Data

We have two CSV files:
- **train.csv** → 1460 houses where we know the actual SalePrice. We use this to *teach* our model.
- **test.csv**  → 1459 houses where SalePrice is hidden. We use this to *test* our model.

> ⚠️ Make sure train.csv and test.csv are in the **same folder** as this notebook.

In [ ]:
# Load both files into dataframes (think: Excel sheets in Python)
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print('Train data shape:', train.shape)   # rows x columns
print('Test data shape: ', test.shape)

# Save the test IDs — we'll need them at the very end for submission
test_ids = test['Id']

In [ ]:
# Look at the first 5 rows of training data
# This is always the first thing to do when you get a new dataset
train.head()

---
## STEP 3 — Explore the Data (EDA)

EDA = Exploratory Data Analysis. Before building any model, we need to **understand** our data.

Key questions to answer:
1. What does our target variable (SalePrice) look like?
2. Which features are most related to SalePrice?
3. Are there any missing values?

In [ ]:
# 3a. Basic statistics about every column
# This shows min, max, mean, count etc. for all numeric columns
train.describe()

In [ ]:
# 3b. Look at the target variable — SalePrice
print('SalePrice Statistics:')
print(f'  Minimum price : ${train["SalePrice"].min():,.0f}')
print(f'  Maximum price : ${train["SalePrice"].max():,.0f}')
print(f'  Average price : ${train["SalePrice"].mean():,.0f}')
print(f'  Median price  : ${train["SalePrice"].median():,.0f}')

# Plot the distribution of SalePrice
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of raw SalePrice
axes[0].hist(train['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of SalePrice')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Number of Houses')

# Right: log-transformed SalePrice (more bell-shaped = better for models)
axes[1].hist(np.log1p(train['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribution of log(SalePrice)')
axes[1].set_xlabel('log(Sale Price)')
axes[1].set_ylabel('Number of Houses')

plt.tight_layout()
plt.show()

# Why log? Machine learning models work better when the target has a bell-shaped distribution.
# The right chart is more bell-shaped, so we'll use log(SalePrice) as our target.

In [ ]:
# 3c. Find which features are most correlated with SalePrice
# Correlation ranges from -1 to 1:
#   1.0  = perfectly related (as X goes up, price goes up)
#  -1.0  = perfectly opposite (as X goes up, price goes down)
#   0.0  = no relationship

# Get correlation of all numeric columns with SalePrice
numeric_corr = train.select_dtypes(include='number').corr()['SalePrice']
top_features = numeric_corr.abs().sort_values(ascending=False).head(11).index

print('Top 10 features most correlated with SalePrice:')
print(numeric_corr[top_features].drop('SalePrice'))

# Visualize as a bar chart
corr_values = numeric_corr[top_features].drop('SalePrice').sort_values()
colors = ['coral' if c < 0 else 'steelblue' for c in corr_values]
corr_values.plot(kind='barh', color=colors)
plt.title('Feature Correlation with SalePrice')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# 3d. Scatter plots of top features vs SalePrice
# This helps us visually see the relationship

top4 = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(top4):
    axes[i].scatter(train[feature], train['SalePrice'], alpha=0.4, color='steelblue')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('SalePrice')
    axes[i].set_title(f'{feature} vs SalePrice')

plt.suptitle('Top Features vs Sale Price', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# OverallQual is the quality rating (1-10). Higher quality = higher price — makes sense!
# GrLivArea is the total above-ground living area in sq ft. Bigger house = higher price!

In [ ]:
# 3e. Check for missing values — very important!
# Missing values (NaN) will cause errors in our model if not handled

missing_train = train.isnull().sum().sort_values(ascending=False)
missing_train = missing_train[missing_train > 0]

print(f'Columns with missing values in train: {len(missing_train)}')
print('\nTop 15 columns with most missing values:')
print(missing_train.head(15))

# Show as percentage
missing_pct = (missing_train / len(train) * 100).round(1)
print('\nAs percentage of total rows:')
print(missing_pct.head(15))

---
## STEP 4 — Clean the Data

Now we fix the problems we found:
1. Drop columns where more than 45% of values are missing (not enough data to be useful)
2. Fill remaining missing values with sensible defaults
3. We combine train and test before cleaning so both get treated the same way

In [ ]:
# 4a. Combine train and test for preprocessing
# This ensures we apply the same cleaning steps to both

# First save the target variable and remove it from train
y_train = np.log1p(train['SalePrice'])   # use log so distribution is nicer
train_size = len(train)

# Stack train (without SalePrice) on top of test
all_data = pd.concat([train.drop('SalePrice', axis=1), test], ignore_index=True)

print('Combined data shape:', all_data.shape)
print('We will split it back later. Train rows: 0 to', train_size - 1)

In [ ]:
# 4b. Drop columns with too many missing values (over 45% missing)

missing_pct_all = all_data.isnull().sum() / len(all_data) * 100
cols_to_drop = missing_pct_all[missing_pct_all > 45].index.tolist()

print(f'Dropping {len(cols_to_drop)} columns with >45% missing values:')
print(cols_to_drop)

all_data = all_data.drop(columns=cols_to_drop)
print(f'\nData shape after dropping: {all_data.shape}')

In [ ]:
# 4c. Fill missing values for remaining columns

# For TEXT columns: fill with 'None' (meaning the feature doesn't exist)
# e.g. GarageType is missing when a house has no garage — 'None' makes sense
text_cols = all_data.select_dtypes(include='object').columns
all_data[text_cols] = all_data[text_cols].fillna('None')

# For NUMBER columns: fill with 0 or the median
# These columns represent measurements — 0 means the feature doesn't exist
zero_fill_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
                  'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                  'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in zero_fill_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(0)

# For all other numeric columns: fill with the median (middle value)
num_cols = all_data.select_dtypes(include='number').columns
all_data[num_cols] = all_data[num_cols].fillna(all_data[num_cols].median())

# Check no missing values remain
remaining_missing = all_data.isnull().sum().sum()
print(f'Total missing values remaining: {remaining_missing}')  # Should be 0

---
## STEP 5 — Feature Engineering

Feature engineering means creating **new columns** from existing ones to give the model
more useful information. For example, total square footage is more useful than
basement + 1st floor + 2nd floor separately.

In [ ]:
# 5a. Create new useful features

# Total square footage (basement + 1st floor + 2nd floor)
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# Total number of bathrooms
all_data['TotalBaths'] = (all_data['FullBath'] +
                          all_data['BsmtFullBath'] +
                          0.5 * all_data['HalfBath'] +
                          0.5 * all_data['BsmtHalfBath'])

# How old was the house when it was sold?
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']

# Years since last renovation
all_data['YearsSinceRemod'] = all_data['YrSold'] - all_data['YearRemodAdd']

# Does the house have a garage? (1 = yes, 0 = no)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)

# Does the house have a basement?
all_data['HasBasement'] = (all_data['TotalBsmtSF'] > 0).astype(int)

print('New features created:')
print(['TotalSF', 'TotalBaths', 'HouseAge', 'YearsSinceRemod', 'HasGarage', 'HasBasement'])
print('\nData shape now:', all_data.shape)

---
## STEP 6 — Encode Categorical (Text) Columns

Machine learning models can only work with **numbers**, not text.
So we need to convert text columns like `Neighborhood = 'Sawyer'` into numbers.

We use **One-Hot Encoding** (pd.get_dummies) which creates a separate 0/1 column
for each unique value. Example:

| Neighborhood | → | N_Sawyer | N_NoRidge | N_OldTown |
|---|---|---|---|---|
| Sawyer | → | 1 | 0 | 0 |
| NoRidge | → | 0 | 1 | 0 |

In [ ]:
# 6a. One-Hot Encode all text columns at once

print('Shape before encoding:', all_data.shape)

all_data = pd.get_dummies(all_data, drop_first=True)

print('Shape after encoding: ', all_data.shape)
print('\nNote: more columns now because each text category became its own column')

In [ ]:
# 6b. Split back into train and test
# Remember we combined them in Step 4 — now we separate them again

X_train = all_data[:train_size]   # first 1460 rows = our training data
X_test  = all_data[train_size:]   # remaining rows = test data

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print('y_train shape:', y_train.shape)

---
## STEP 7 — Train Machine Learning Models

We'll try 3 models:
1. **Linear Regression** — the simplest model, good baseline
2. **Random Forest** — uses many decision trees, usually much better
3. **Gradient Boosting** — the most powerful, often wins competitions

We'll use **cross-validation** to fairly measure each model's accuracy.
Cross-validation splits the training data into 5 parts, trains on 4 parts,
tests on the 5th, and repeats 5 times — giving a reliable score.

In [ ]:
# 7a. Helper function to evaluate any model

def evaluate_model(model, X, y, model_name):
    """
    Trains the model using cross-validation and prints the RMSE score.
    RMSE = Root Mean Squared Error. Lower is better.
    Since we used log(SalePrice), the RMSE is in log scale (~0.12 is good).
    """
    # neg_mean_squared_error returns negative values, so we negate it
    scores = cross_val_score(model, X, y,
                             scoring='neg_mean_squared_error',
                             cv=5)  # 5-fold cross validation
    rmse_scores = np.sqrt(-scores)
    print(f'{model_name}:')
    print(f'  Mean RMSE : {rmse_scores.mean():.4f}')
    print(f'  Std RMSE  : {rmse_scores.std():.4f}  (lower std = more consistent)')
    print()
    return rmse_scores.mean()

print('Evaluation function defined. Ready to train!')

In [ ]:
# 7b. Model 1: Linear Regression
# The simplest model — it fits a straight line through the data

lr_model = LinearRegression()
lr_score = evaluate_model(lr_model, X_train, y_train, 'Linear Regression')

In [ ]:
# 7c. Model 2: Random Forest
# Builds 200 decision trees and averages their predictions
# n_estimators = number of trees; more trees = better but slower
# random_state = ensures you get the same result each time you run it

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_score = evaluate_model(rf_model, X_train, y_train, 'Random Forest')

In [ ]:
# 7d. Model 3: Gradient Boosting
# Builds trees one by one, each one correcting the mistakes of the previous
# learning_rate = how much each tree corrects (lower = more careful)
# n_estimators = how many trees
# max_depth = how deep/complex each tree is

gb_model = GradientBoostingRegressor(n_estimators=300,
                                      learning_rate=0.05,
                                      max_depth=4,
                                      random_state=42)
gb_score = evaluate_model(gb_model, X_train, y_train, 'Gradient Boosting')

In [ ]:
# 7e. Compare all models visually

model_names = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
scores = [lr_score, rf_score, gb_score]
colors = ['#a8c6e8', '#f4a261', '#2a9d8f']

bars = plt.bar(model_names, scores, color=colors, edgecolor='white', linewidth=1.5)
plt.ylabel('RMSE (lower is better)')
plt.title('Model Comparison — Cross-Validation RMSE')

# Add score labels on top of bars
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.001,
             f'{score:.4f}',
             ha='center', va='bottom', fontweight='bold')

plt.ylim(0, max(scores) * 1.2)
plt.tight_layout()
plt.show()

best_model_name = model_names[np.argmin(scores)]
print(f'Best model: {best_model_name} with RMSE = {min(scores):.4f}')

---
## STEP 8 — Feature Importance

Random Forest and Gradient Boosting can tell us **which features matter most**.
This is very useful — it tells you what drives house prices.

In [ ]:
# 8a. Train the best model on the full training data
# (Cross-validation uses only part of the data each time.
#  Now we train on ALL training data for the final model.)

gb_model.fit(X_train, y_train)
print('Model trained on full training data!')

In [ ]:
# 8b. Plot the top 20 most important features

feature_importance = pd.Series(
    gb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(20)

feature_importance.sort_values().plot(kind='barh', color='#2a9d8f')
plt.title('Top 20 Most Important Features for Predicting House Price')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
for i, (feat, score) in enumerate(feature_importance.head(5).items(), 1):
    print(f'  {i}. {feat}: {score:.4f}')

---
## STEP 9 — Make Predictions and Create Submission File

Now we use our trained model to predict prices for the test set
and save the results in the format required for submission.

In [ ]:
# 9a. Predict on test data
# Remember: our model predicts log(SalePrice)
# So we need to reverse the log with np.expm1() to get actual dollar amounts

log_predictions = gb_model.predict(X_test)
predictions = np.expm1(log_predictions)   # reverse log transformation

print('Sample predictions (first 5 houses):')
for i, price in enumerate(predictions[:5]):
    print(f'  House {i+1}: ${price:,.0f}')

In [ ]:
# 9b. Create the submission file
# Kaggle requires exactly 2 columns: Id and SalePrice

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission.to_csv('submission.csv', index=False)

print('submission.csv saved!')
print(f'Total predictions: {len(submission)}')
print('\nFirst 5 rows of submission file:')
print(submission.head())

In [ ]:
# 9c. Plot predicted vs actual prices (on training data) to see how well we did

train_predictions = np.expm1(gb_model.predict(X_train))
actual_prices = np.expm1(y_train)

plt.figure(figsize=(10, 8))
plt.scatter(actual_prices, train_predictions, alpha=0.4, color='steelblue')

# Perfect prediction line
max_val = max(actual_prices.max(), train_predictions.max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')

plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.title('Actual vs Predicted Sale Price (Training Data)')
plt.legend()
plt.tight_layout()
plt.show()

# Points close to the red line = accurate predictions

---
## STEP 10 — Summary & What to Improve

### What we did:
1. ✅ Loaded and explored the dataset (1460 houses, 81 features)
2. ✅ Cleaned missing values
3. ✅ Created new useful features (TotalSF, HouseAge, TotalBaths…)
4. ✅ Converted text columns to numbers
5. ✅ Compared 3 models (Linear Regression, Random Forest, Gradient Boosting)
6. ✅ Generated a submission file

### How to improve your score:
- Try **XGBoost** (`pip install xgboost`) — often the best for tabular data
- Try **blending** predictions: `final = 0.5 * rf_preds + 0.5 * gb_preds`
- Remove **outliers**: very large houses with low prices confuse the model
- Add more **feature engineering** (e.g. price per sq ft, total porch area)
- Tune **hyperparameters** using GridSearchCV

In [ ]:
print('=== TASK COMPLETE ===')
print(f'Your submission file is ready: submission.csv')
print(f'Upload it to Kaggle to see your leaderboard score!')